# 01. Value Based RL Dqn Rainbow and Qr Dqn

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Watkins (1989 Q-Learning); Mnih et al. (2015 Nature DQN); van Hasselt et al. (2016 Double DQN); Wang et al. (2016 Dueling); Schaul et al. (2016 PER); Dabney et al. (2018 QR-DQN); Hessel et al. (2018 Rainbow).
- **Purpose:** First-principles implementation and verification of foundational value-based reinforcement learning algorithms: Double DQN, Dueling Architecture, Prioritized Experience Replay (PER), and Distributional Quantile Regression (QR-DQN).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Active Learning Bald Coreset and Badge](../01_self_semi_meta_continual_federated/07_active_learning_bald_coreset_and_badge.ipynb) $\to$ `🔗 Downstream Syntheses:` [Policy Gradient Reinforce A2c PPO and GFlowNets](02_policy_gradient_reinforce_a2c_ppo_and_gflownets.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Do not use discrete value-based RL when the action space is continuous (use SAC or TD3 instead). Avoid vanilla single-network DQN on environments with stochastic transitions or rewards, as Jensen's inequality and maximization bias $\mathbb{E}[\max X] \ge \max \mathbb{E}[X]$ cause catastrophic value overestimation. Never use Prioritized Experience Replay without importance-sampling weight correction ($\beta > 0$), as non-uniform transition replay severely distorts the expected stochastic gradient.

## Mathematical Foundations & Governing Equations

### 1. Double DQN Decoupled Bellman Targets
$$
a^* = \arg\max_{a'} Q_\theta(s', a'), \quad y^{\text{Double}} = r + \gamma (1 - d) Q_{\bar{\theta}}(s', a^*)
$$

### 2. Dueling Architecture
$$
Q(s, a) = V(s) + \left( A(s, a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a') \right)
$$

### 3. Prioritized Experience Replay (PER)
$$
P(i) = \frac{|\delta_i|^\alpha + \epsilon}{\sum_k (|\delta_k|^\alpha + \epsilon)}, \quad w_i = \left( \frac{1}{N \cdot P(i)} \right)^\beta, \quad \tilde{w}_i = \frac{w_i}{\max_j w_j}
$$
$$
\mathcal{L}_{\text{PER}} = \frac{1}{B} \sum_{i=1}^B \tilde{w}_i \cdot (Q(s_i, a_i) - y_i)^2
$$

### 4. Quantile Regression (QR-DQN)
$$
\tau_i = \frac{2i - 1}{2N}, \quad \rho_{\tau_i}^\kappa(u) = |\tau_i - \mathbb{I}(u < 0)| \cdot \frac{\mathcal{L}_\kappa(u)}{\kappa}
$$
$$
\mathcal{L}_{\text{QR}} = \frac{1}{B N^2} \sum_{b=1}^B \sum_{i=1}^N \sum_{j=1}^N \rho_{\tau_i}^\kappa \left( \hat{\mathcal{T}}\theta_j(s_b', a^*) - \theta_i(s_b, a_b) \right)
$$

### Asymptotic Time & Memory Complexity
- Double DQN Target Selection: $\mathcal{O}(B \cdot |\mathcal{A}|)$ time, $\mathcal{O}(B)$ memory.
- Dueling Q Computation: $\mathcal{O}(B \cdot |\mathcal{A}|)$ time, $\mathcal{O}(B \cdot |\mathcal{A}|)$ memory.
- Quantile Huber Loss: $\mathcal{O}(B \cdot N^2)$ time and memory for $N$ support quantiles.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint from repo root or parents
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def dueling_q_values(value: torch.Tensor, advantage: torch.Tensor) -> torch.Tensor:
    mean_adv = torch.mean(advantage, dim=-1, keepdim=True)
    return value + (advantage - mean_adv)

def double_dqn_targets(
    rewards: torch.Tensor,
    dones: torch.Tensor,
    q_next_online: torch.Tensor,
    q_next_target: torch.Tensor,
    gamma: float = 0.99,
) -> torch.Tensor:
    best_actions = torch.argmax(q_next_online, dim=-1, keepdim=True)
    q_target_selected = torch.gather(q_next_target, dim=-1, index=best_actions).squeeze(-1)
    return rewards + gamma * (1.0 - dones) * q_target_selected

def per_loss(
    q_eval: torch.Tensor,
    targets: torch.Tensor,
    is_weights: torch.Tensor,
) -> torch.Tensor:
    td_errors = q_eval - targets
    weighted_sq_errors = is_weights * (td_errors ** 2)
    return torch.mean(weighted_sq_errors)

def quantile_huber_loss(
    current_quantiles: torch.Tensor,
    target_quantiles: torch.Tensor,
    tau: torch.Tensor,
    kappa: float = 1.0,
) -> torch.Tensor:
    diff = target_quantiles.unsqueeze(1) - current_quantiles.unsqueeze(2)
    abs_diff = torch.abs(diff)
    huber = torch.where(abs_diff <= kappa, 0.5 * (diff ** 2), kappa * (abs_diff - 0.5 * kappa))
    tau_expanded = tau.view(1, -1, 1)
    indicator = (diff < 0.0).float()
    weight = torch.abs(tau_expanded - indicator)
    return torch.mean(weight * huber / kappa)

In [ ]:
# Numerical parity verification across all components
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-6, f"Linear anchor parity error: {err_anchor:.2e}"

q_duel = dueling_q_values(weights["value"], weights["advantage"])
err_duel = float(torch.max(torch.abs(q_duel - weights["dueling_q_ref"])))
assert err_duel < 1e-6, f"Dueling Q parity error: {err_duel:.2e}"

ddqn_targets = double_dqn_targets(
    weights["rewards"], weights["dones"], weights["q_next_online"], weights["q_next_target"], gamma=0.99
)
err_ddqn = float(torch.max(torch.abs(ddqn_targets - weights["ddqn_targets_ref"])))
assert err_ddqn < 1e-6, f"Double DQN targets parity error: {err_ddqn:.2e}"

per_l = per_loss(weights["q_eval"], weights["targets"], weights["is_weights"])
err_per = float(torch.max(torch.abs(per_l - weights["per_loss_ref"])))
assert err_per < 1e-6, f"PER loss parity error: {err_per:.2e}"

qr_l = quantile_huber_loss(weights["current_quantiles"], weights["target_quantiles"], weights["tau"], kappa=1.0)
err_qr = float(torch.max(torch.abs(qr_l - weights["qr_loss_ref"])))
assert err_qr < 1e-6, f"QR loss parity error: {err_qr:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_duel, err_ddqn, err_per, err_qr):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_duel = benchmark_fn(
    lambda: dueling_q_values(weights["value"], weights["advantage"]),
    iters=100,
    name="Dueling Q Aggregation",
)
b_ddqn = benchmark_fn(
    lambda: double_dqn_targets(weights["rewards"], weights["dones"], weights["q_next_online"], weights["q_next_target"]),
    iters=100,
    name="Double DQN Targets",
)
b_qr = benchmark_fn(
    lambda: quantile_huber_loss(weights["current_quantiles"], weights["target_quantiles"], weights["tau"]),
    iters=100,
    name="QR-DQN Huber Loss",
)

print(format_benchmark_table([b_duel, b_ddqn, b_qr]))

In [ ]:
# Failure Mode Contrast: Maximization Bias in Standard DQN vs Double DQN
# In a stochastic environment where true Q* is 0 for all actions,
# but sample estimates have zero-mean Gaussian noise N(0, sigma^2).
torch.manual_seed(42)
num_trials = 1000
num_actions = 10
noise = torch.randn(num_trials, num_actions) * 1.5

# Standard DQN: max_a Q(s', a) overestimates true value
vanilla_dqn_estimate = torch.max(noise, dim=-1).values.mean().item()

# Double DQN: action selected by online Q, evaluated by independent target Q
online_noise = noise
target_noise = torch.randn(num_trials, num_actions) * 1.5
selected_actions = torch.argmax(online_noise, dim=-1, keepdim=True)
double_dqn_estimate = torch.gather(target_noise, dim=-1, index=selected_actions).mean().item()

print(f"Vanilla DQN Expected Return Estimate: {vanilla_dqn_estimate:.4f} (Severe upward bias! True value is 0.0)")
print(f"Double DQN Expected Return Estimate:  {double_dqn_estimate:.4f} (Unbiased near 0.0)")

# Assert that vanilla DQN exhibits upward maximization bias while Double DQN suppresses it
assert vanilla_dqn_estimate > 1.0, "Vanilla DQN must demonstrate strong positive maximization bias"
assert abs(double_dqn_estimate) < 0.2, "Double DQN estimate must remain near true zero mean"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Decoupling Selection and Evaluation**: Double DQN eliminates maximization bias by having the online network pick actions while the target network scores them.
2. **Identifiability in Dueling Networks**: Mean subtraction $(A - \bar{A})$ forces the value stream $V(s)$ to genuinely reflect the scalar baseline state value without arbitrary constant offsets.
3. **Distributional Modeling**: QR-DQN models the full return distribution via quantile regression, capturing multimodal outcomes and tail risks that mean-value RL obscures.

---
### Cross-Domain Generalisation Matrix
- **Finance**: Quantile value estimation allows risk-sensitive portfolio execution directly bounded by Value-at-Risk (VaR) or Conditional VaR (CVaR).
- **Robotics**: Dueling decomposition separates stable robot posture/orientation balance $V(s)$ from micro-actuator torque selection $A(s, a)$.
- **Autonomous Vehicles**: Distributional returns prevent autonomous planners from averaging fatal catastrophic collision returns into benign near-miss returns.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_value_based_rl_dqn_rainbow_and_qr_dqn.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.md)
- **PyTorch 2.x Module:** [01_value_based_rl_dqn_rainbow_and_qr_dqn.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.py)
- **Functional JAX Module:** [01_value_based_rl_dqn_rainbow_and_qr_dqn.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.py)
- **Native Rust Candle Track:** [01_value_based_rl_dqn_rainbow_and_qr_dqn.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.rs)
- **Hardware Kernel (CPU / WGSL):** [01_value_based_rl_dqn_rainbow_and_qr_dqn.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.rs)
- **Golden SafeTensors:** [01_value_based_rl_dqn_rainbow_and_qr_dqn.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/01_value_based_rl_dqn_rainbow_and_qr_dqn.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Active Learning Bald Coreset and Badge](../01_self_semi_meta_continual_federated/07_active_learning_bald_coreset_and_badge.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 02. Policy Gradient Reinforce A2c PPO and GFlowNets](02_policy_gradient_reinforce_a2c_ppo_and_gflownets.ipynb) &rarr;